# Neural Embedding Search

We build a semantic search engine using neural embeddings.

Unlike TF-IDF, this method captures meaning, not just keyword overlap.


In [1]:
import pandas as pd
import sys
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

print(sys.executable)

c:\Users\Felhasználó\TextMiningAndNaturalLanguageProcessingHomeAssignment\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


c:\Users\Felhasználó\TextMiningAndNaturalLanguageProcessingHomeAssignment\.venv\Scripts\python.exe


In [2]:
df = pd.read_csv("../Data/final_tweets.csv")

print(df.shape)
df.head()

(59184, 4)


,tweet_id,text,clean_text,company
0,277379,Another great set of @Delta flights thank to #...,another great set of flights thank to tmobilew...,delta
1,2488664,@AppleSupport why is my iPhone automatically g...,why is my iphone automatically going on mute,applesupport
2,2383194,@Uber_Support Since yesterday I haven't receiv...,since yesterday i havent received any update a...,uber_support
3,1610886,"@AmazonHelp There is no email from 2 days, jus...",there is no email from days just asked to wait...,amazonhelp
4,2763258,"@SouthwestAir thanks for responding, will call...",thanks for responding will call as soon as i g...,southwestair


In [3]:
model = SentenceTransformer("all-MiniLM-L6-v2")

print("Model loaded successfully")

c:\Users\Felhasználó\TextMiningAndNaturalLanguageProcessingHomeAssignment\.venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Felhasználó\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 103/103 [0

Model loaded successfully


In [4]:
search_df = df[["tweet_id", "clean_text", "company"]].copy()

search_df.head()

,tweet_id,clean_text,company
0,277379,another great set of flights thank to tmobilew...,delta
1,2488664,why is my iphone automatically going on mute,applesupport
2,2383194,since yesterday i havent received any update a...,uber_support
3,1610886,there is no email from days just asked to wait...,amazonhelp
4,2763258,thanks for responding will call as soon as i g...,southwestair


In [5]:
texts = search_df["clean_text"].tolist()

embeddings = model.encode(texts, show_progress_bar=True)

print("Embeddings shape:", embeddings.shape)

Batches: 100%|██████████| 1850/1850 [06:02<00:00,  5.10it/s]


Embeddings shape: (59184, 384)


In [11]:
def semantic_search(query, top_k=5):
    # embed query
    query_vec = model.encode([query],normalize_embeddings=True)
    
    # compute similarity
    scores = cosine_similarity(query_vec, embeddings).flatten()
    
    # get top results
    top_indices = scores.argsort()[::-1][:top_k]
    
    results = search_df.iloc[top_indices].copy()
    results["score"] = scores[top_indices]
    
    return results

In [12]:
queries = [
    "flight delayed customer service",
    "refund for cancelled order",
    "app not working",
    "bad customer support",
    "lost baggage complaint"
]

for q in queries:
    print("=" * 80)
    print("Query:", q)
    display(semantic_search(q, top_k=5))

Query: flight delayed customer service


,tweet_id,clean_text,company,score
2472,1799160,flight was already delayed on top of it poor c...,americanair,0.856027
41166,2721339,flight delayed and theres a hr wait on custome...,delta,0.789575
24784,366638,flight delayed hours but no texts no emails no...,delta,0.776327
46536,2258648,been there done that with customer service thr...,americanair,0.773329
36349,2098673,my flight has been delayed hours and no custom...,americanair,0.769100


Query: refund for cancelled order


,tweet_id,clean_text,company,score
22077,2283942,i cancelled an order and it successfully cance...,amazonhelp,0.820157
45112,2575420,ee cancel my order refund my money,uber_support,0.795096
25933,581913,ur representative misguided me against the amo...,amazonhelp,0.790322
44897,1736159,order cancelled,amazonhelp,0.770140
29980,242928,cancelled my order,amazonhelp,0.762320


Query: app not working


,tweet_id,clean_text,company,score
2469,2843793,i uninstalled the app and now its working,amazonhelp,0.771482
50622,1177828,redownload the app now its working again,spotifycares,0.768622
33769,992114,your app is broken again,delta,0.767871
37962,988460,is something goin on with the app i cant get a...,southwestair,0.715760
38151,2409071,i cant even see on the app,uber_support,0.691451


Query: bad customer support


,tweet_id,clean_text,company,score
1329,1098833,worst customer support ever,delta,0.869834
287,382575,customer for years worst customer support ever,amazonhelp,0.845081
55640,1007281,terrible customer service,americanair,0.802420
36904,2467779,customer service is terrible,delta,0.771772
843,2123534,terrible customer service once again,delta,0.769033


Query: lost baggage complaint


,tweet_id,clean_text,company,score
57075,1906961,could you advise me on my lost baggage claim,delta,0.727050
50273,630078,the lost baggage process and the people who ma...,delta,0.705007
31128,2009442,your handling of my missing luggage is ridicul...,americanair,0.701697
58694,2268208,does ur complains procedure include ignoring c...,americanair,0.699116
5558,2254805,the baggage service agents shirley viola and s...,southwestair,0.697405


The neural embedding search gave good results for all five queries. It found tweets with similar meanings, not just the same exact words.

For example, “bad customer support” also found tweets about “terrible customer service” and “worst customer support.” This shows that the model can understand similar complaints.

Overall, embedding search is useful because people write complaints in different ways, and it can still find related tweets.